# 06 - Backend Integration

## CyberForge AI - API-Ready Model Packaging

This notebook prepares models for backend integration:
- Model serialization in API-friendly formats
- Versioned model artifacts
- Backend-compatible inference endpoints
- Integration with mlService.js and ThreatService.js

### Backend Integration Points:
- `mlService.js` - Primary ML model interface
- `WebScraperAPIService.js` - Input data format
- `threatService.js` - Threat analysis outputs

In [ ]:
import json
import os
import time
import shutil
from pathlib import Path
from typing import Dict, List, Any
import joblib
import hashlib
import warnings
warnings.filterwarnings('ignore')

# Configuration
config_path = Path("../notebook_config.json")
with open(config_path) as f:
    CONFIG = json.load(f)

MODELS_DIR = Path(CONFIG["datasets_dir"]).parent / "models"
AGENT_DIR = MODELS_DIR.parent / "agent"
BACKEND_DIR = MODELS_DIR.parent / "backend_package"
BACKEND_DIR.mkdir(exist_ok=True)

print(f"✓ Configuration loaded")
print(f"✓ Backend package output: {BACKEND_DIR}")

## 1. Define Backend API Contracts

In [ ]:
class BackendAPIContracts:
    """
    Define API contracts matching backend services.
    Aligned with mlService.js and threatService.js
    """
    
    # Request format from WebScraperAPIService
    SCRAPER_INPUT = {
        'url': 'string',
        'security_report': {
            'is_https': 'boolean',
            'mixed_content': 'boolean',
            'insecure_cookies': 'boolean',
            'security_headers': 'object'
        },
        'network_requests': 'array',
        'console_logs': 'array',
        'page_content': 'string'
    }
    
    # Response format for mlService.js
    ML_RESPONSE = {
        'prediction': 'number',  # 0 or 1
        'confidence': 'number',  # 0.0 to 1.0
        'risk_level': 'string',  # critical, high, medium, low, info
        'model_name': 'string',
        'model_version': 'string',
        'inference_time_ms': 'number',
        'details': 'object'
    }
    
    # Response format for threatService.js
    THREAT_RESPONSE = {
        'threat_detected': 'boolean',
        'threat_type': 'string',
        'risk_score': 'number',
        'indicators': 'array',
        'recommended_action': 'string',
        'reasoning': 'string'
    }
    
    @classmethod
    def format_ml_response(cls, prediction: int, confidence: float, 
                           model_name: str, inference_time: float,
                           details: Dict = None) -> Dict:
        """Format response for mlService.js"""
        risk_level = (
            'critical' if confidence >= 0.9 else
            'high' if confidence >= 0.7 else
            'medium' if confidence >= 0.5 else
            'low' if confidence >= 0.3 else 'info'
        )
        
        return {
            'prediction': int(prediction),
            'confidence': float(confidence),
            'risk_level': risk_level,
            'model_name': model_name,
            'model_version': '1.0.0',
            'inference_time_ms': float(inference_time),
            'details': details or {}
        }
    
    @classmethod
    def format_threat_response(cls, detected: bool, threat_type: str,
                               score: float, indicators: List,
                               action: str, reasoning: str) -> Dict:
        """Format response for threatService.js"""
        return {
            'threat_detected': detected,
            'threat_type': threat_type,
            'risk_score': float(score),
            'indicators': indicators,
            'recommended_action': action,
            'reasoning': reasoning
        }

print("✓ Backend API Contracts defined")

## 2. Model Packager

In [ ]:
class ModelPackager:
    """
    Package models for backend deployment.
    Creates versioned, self-contained model artifacts.
    """
    
    def __init__(self, models_dir: Path, output_dir: Path):
        self.models_dir = models_dir
        self.output_dir = output_dir
        self.package_manifest = {'models': {}, 'version': '1.0.0'}
    
    def calculate_checksum(self, file_path: Path) -> str:
        """Calculate MD5 checksum for file integrity"""
        with open(file_path, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    
    def package_model(self, model_name: str, model_info: Dict) -> Dict:
        """Package a single model for backend"""
        model_type = model_info.get('best_model', 'random_forest')
        source_dir = self.models_dir / model_name
        dest_dir = self.output_dir / model_name
        dest_dir.mkdir(exist_ok=True)
        
        packaged_files = {}
        
        # Copy model file
        model_source = source_dir / f"{model_type}.pkl"
        if model_source.exists():
            model_dest = dest_dir / "model.pkl"
            shutil.copy(model_source, model_dest)
            packaged_files['model'] = {
                'path': str(model_dest.relative_to(self.output_dir)),
                'checksum': self.calculate_checksum(model_dest),
                'size_bytes': model_dest.stat().st_size
            }
        
        # Copy scaler
        scaler_source = source_dir / "scaler.pkl"
        if scaler_source.exists():
            scaler_dest = dest_dir / "scaler.pkl"
            shutil.copy(scaler_source, scaler_dest)
            packaged_files['scaler'] = {
                'path': str(scaler_dest.relative_to(self.output_dir)),
                'checksum': self.calculate_checksum(scaler_dest)
            }
        
        # Copy metadata
        meta_source = source_dir / f"{model_type}_metadata.json"
        if meta_source.exists():
            meta_dest = dest_dir / "metadata.json"
            shutil.copy(meta_source, meta_dest)
            packaged_files['metadata'] = {
                'path': str(meta_dest.relative_to(self.output_dir))
            }
        
        # Create model info
        model_pkg_info = {
            'name': model_name,
            'type': model_type,
            'version': '1.0.0',
            'accuracy': model_info.get('accuracy', 0),
            'f1_score': model_info.get('f1_score', 0),
            'inference_time_ms': model_info.get('inference_time_ms', 0),
            'files': packaged_files,
            'packaged_at': time.strftime('%Y-%m-%d %H:%M:%S')
        }
        
        # Save model info
        info_path = dest_dir / "package_info.json"
        with open(info_path, 'w') as f:
            json.dump(model_pkg_info, f, indent=2)
        
        self.package_manifest['models'][model_name] = model_pkg_info
        
        return model_pkg_info
    
    def save_manifest(self):
        """Save package manifest"""
        manifest_path = self.output_dir / "manifest.json"
        self.package_manifest['created_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
        with open(manifest_path, 'w') as f:
            json.dump(self.package_manifest, f, indent=2)
        return manifest_path

packager = ModelPackager(MODELS_DIR, BACKEND_DIR)
print("✓ Model Packager initialized")

## 3. Package Models

In [ ]:
# Load model registry
registry_path = MODELS_DIR / "model_registry.json"

if registry_path.exists():
    with open(registry_path) as f:
        registry = json.load(f)
    print(f"✓ Loaded {len(registry.get('models', {}))} models")
else:
    registry = {'models': {}}
    print("⚠ No registry found")

# Package each model
print("\nPackaging models for backend...\n")

for model_name, model_info in registry.get('models', {}).items():
    print(f"  Packaging: {model_name}")
    try:
        pkg_info = packager.package_model(model_name, model_info)
        print(f"    ✓ Files: {len(pkg_info['files'])}")
        print(f"    ✓ Version: {pkg_info['version']}")
    except Exception as e:
        print(f"    ⚠ Error: {e}")

# Save manifest
manifest_path = packager.save_manifest()
print(f"\n✓ Manifest saved to: {manifest_path}")

## 4. Generate Backend Integration Code

In [ ]:
# Generate Python inference module for backend
inference_module = '''
"""
CyberForge ML Inference Module
Backend integration for mlService.js
"""

import json
import time
import joblib
import numpy as np
from pathlib import Path
from typing import Dict, List, Any, Optional

class CyberForgeInference:
    """
    ML inference service for CyberForge backend.
    Compatible with mlService.js API contract.
    """
    
    def __init__(self, models_dir: str):
        self.models_dir = Path(models_dir)
        self.loaded_models = {}
        self.manifest = self._load_manifest()
    
    def _load_manifest(self) -> Dict:
        manifest_path = self.models_dir / "manifest.json"
        if manifest_path.exists():
            with open(manifest_path) as f:
                return json.load(f)
        return {"models": {}}
    
    def load_model(self, model_name: str) -> bool:
        """Load a model into memory"""
        if model_name in self.loaded_models:
            return True
        
        model_dir = self.models_dir / model_name
        model_path = model_dir / "model.pkl"
        scaler_path = model_dir / "scaler.pkl"
        
        if not model_path.exists():
            return False
        
        self.loaded_models[model_name] = {
            "model": joblib.load(model_path),
            "scaler": joblib.load(scaler_path) if scaler_path.exists() else None
        }
        return True
    
    def predict(self, model_name: str, features: Dict) -> Dict:
        """
        Make a prediction.
        
        Args:
            model_name: Name of the model to use
            features: Feature dictionary
            
        Returns:
            Response matching mlService.js contract
        """
        if not self.load_model(model_name):
            return {"error": f"Model not found: {model_name}"}
        
        model_data = self.loaded_models[model_name]
        model = model_data["model"]
        scaler = model_data["scaler"]
        
        # Convert features to array
        X = np.array([list(features.values())])
        
        # Scale if scaler available
        if scaler:
            X = scaler.transform(X)
        
        # Predict
        start_time = time.time()
        prediction = int(model.predict(X)[0])
        inference_time = (time.time() - start_time) * 1000
        
        # Get confidence
        confidence = 0.5
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X)[0]
            confidence = float(max(proba))
        
        # Determine risk level
        risk_level = (
            "critical" if confidence >= 0.9 else
            "high" if confidence >= 0.7 else
            "medium" if confidence >= 0.5 else
            "low" if confidence >= 0.3 else "info"
        )
        
        return {
            "prediction": prediction,
            "confidence": confidence,
            "risk_level": risk_level,
            "model_name": model_name,
            "model_version": "1.0.0",
            "inference_time_ms": inference_time
        }
    
    def batch_predict(self, model_name: str, features_list: List[Dict]) -> List[Dict]:
        """Batch predictions"""
        return [self.predict(model_name, f) for f in features_list]
    
    def list_models(self) -> List[str]:
        """List available models"""
        return list(self.manifest.get("models", {}).keys())
    
    def get_model_info(self, model_name: str) -> Dict:
        """Get model information"""
        return self.manifest.get("models", {}).get(model_name, {})


# FastAPI integration
def create_api(models_dir: str):
    """Create FastAPI app for model serving"""
    try:
        from fastapi import FastAPI, HTTPException
        from pydantic import BaseModel
    except ImportError:
        return None
    
    app = FastAPI(title="CyberForge ML API", version="1.0.0")
    inference = CyberForgeInference(models_dir)
    
    class PredictRequest(BaseModel):
        model_name: str
        features: Dict
    
    @app.post("/predict")
    async def predict(request: PredictRequest):
        result = inference.predict(request.model_name, request.features)
        if "error" in result:
            raise HTTPException(status_code=404, detail=result["error"])
        return result
    
    @app.get("/models")
    async def list_models():
        return {"models": inference.list_models()}
    
    @app.get("/models/{model_name}")
    async def get_model_info(model_name: str):
        info = inference.get_model_info(model_name)
        if not info:
            raise HTTPException(status_code=404, detail="Model not found")
        return info
    
    return app


if __name__ == "__main__":
    import sys
    models_dir = sys.argv[1] if len(sys.argv) > 1 else "."
    
    inference = CyberForgeInference(models_dir)
    print(f"Available models: {inference.list_models()}")
'''

inference_path = BACKEND_DIR / "inference.py"
with open(inference_path, 'w') as f:
    f.write(inference_module)

print(f"✓ Inference module saved to: {inference_path}")

In [ ]:
# Generate JavaScript client for backend
js_client = '''
/**
 * CyberForge ML Client
 * Integration with mlService.js
 */

const axios = require('axios');

class CyberForgeMLClient {
    constructor(baseUrl = 'http://localhost:8001') {
        this.baseUrl = baseUrl;
        this.client = axios.create({
            baseURL: baseUrl,
            timeout: 5000,
            headers: { 'Content-Type': 'application/json' }
        });
    }

    /**
     * Get prediction from ML model
     * @param {string} modelName - Name of the model
     * @param {Object} features - Feature dictionary
     * @returns {Promise<Object>} Prediction result
     */
    async predict(modelName, features) {
        try {
            const response = await this.client.post('/predict', {
                model_name: modelName,
                features: features
            });
            return response.data;
        } catch (error) {
            console.error('ML prediction error:', error.message);
            throw error;
        }
    }

    /**
     * Analyze website for threats
     * @param {string} url - URL to analyze
     * @param {Object} scrapedData - Data from WebScraperAPIService
     * @returns {Promise<Object>} Threat analysis result
     */
    async analyzeWebsite(url, scrapedData) {
        try {
            const response = await this.client.post('/analyze', {
                url: url,
                data: scrapedData
            });
            return response.data;
        } catch (error) {
            console.error('Website analysis error:', error.message);
            throw error;
        }
    }

    /**
     * List available models
     * @returns {Promise<Array>} List of model names
     */
    async listModels() {
        const response = await this.client.get('/models');
        return response.data.models;
    }

    /**
     * Get model information
     * @param {string} modelName - Name of the model
     * @returns {Promise<Object>} Model metadata
     */
    async getModelInfo(modelName) {
        const response = await this.client.get(`/models/${modelName}`);
        return response.data;
    }
}

module.exports = CyberForgeMLClient;
'''

js_client_path = BACKEND_DIR / "ml_client.js"
with open(js_client_path, 'w') as f:
    f.write(js_client)

print(f"✓ JavaScript client saved to: {js_client_path}")

## 5. Copy Agent Module

In [ ]:
# Copy agent module to backend package
agent_source = AGENT_DIR / "cyberforge_agent.py"
agent_config_source = AGENT_DIR / "agent_config.json"

if agent_source.exists():
    shutil.copy(agent_source, BACKEND_DIR / "cyberforge_agent.py")
    print(f"✓ Agent module copied")

if agent_config_source.exists():
    shutil.copy(agent_config_source, BACKEND_DIR / "agent_config.json")
    print(f"✓ Agent config copied")

## 6. Generate README

In [ ]:
readme_content = f"""
# CyberForge ML Backend Package

Production-ready ML models for CyberForge backend integration.

## Contents

- `inference.py` - Python inference module
- `ml_client.js` - JavaScript client for Node.js backend
- `cyberforge_agent.py` - Agent intelligence module
- `manifest.json` - Model registry and metadata
- `*/model.pkl` - Trained model files
- `*/scaler.pkl` - Feature scalers
- `*/metadata.json` - Model metadata

## Quick Start

### Python

```python
from inference import CyberForgeInference

inference = CyberForgeInference('./backend_package')
result = inference.predict('phishing_detection', {{'url_length': 50, ...}})
print(result)
```

### Node.js

```javascript
const CyberForgeMLClient = require('./ml_client');

const client = new CyberForgeMLClient('http://localhost:8001');
const result = await client.predict('phishing_detection', {{url_length: 50}});
console.log(result);
```

### FastAPI Server

```bash
pip install fastapi uvicorn
uvicorn inference:create_api --host 0.0.0.0 --port 8001
```

## API Contract

### Prediction Response

```json
{{
    "prediction": 0,
    "confidence": 0.95,
    "risk_level": "low",
    "model_name": "phishing_detection",
    "model_version": "1.0.0",
    "inference_time_ms": 2.5
}}
```

## Models Included

| Model | Type | Accuracy | F1 Score |
|-------|------|----------|----------|
"""

# Add model table
for model_name, model_info in packager.package_manifest.get('models', {}).items():
    readme_content += f"| {model_name} | {model_info.get('type', 'N/A')} | {model_info.get('accuracy', 0):.4f} | {model_info.get('f1_score', 0):.4f} |\n"

readme_content += f"""

## Version

- Package Version: 1.0.0
- Created: {time.strftime('%Y-%m-%d %H:%M:%S')}
"""

readme_path = BACKEND_DIR / "README.md"
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"✓ README saved to: {readme_path}")

## 7. Summary

In [ ]:
# List packaged files
packaged_files = list(BACKEND_DIR.rglob('*'))
total_size = sum(f.stat().st_size for f in packaged_files if f.is_file())

print("\n" + "=" * 60)
print("BACKEND INTEGRATION COMPLETE")
print("=" * 60)

print(f"""
📦 Backend Package:
   - Location: {BACKEND_DIR}
   - Files: {len([f for f in packaged_files if f.is_file()])}
   - Total size: {total_size / (1024*1024):.2f} MB

📁 Package Contents:""")

for f in sorted(BACKEND_DIR.iterdir()):
    if f.is_file():
        print(f"   - {f.name}")
    elif f.is_dir():
        print(f"   - {f.name}/")

print(f"""
🔌 Integration Points:
   - Python: inference.py (CyberForgeInference class)
   - Node.js: ml_client.js (CyberForgeMLClient class)
   - FastAPI: inference.py (create_api function)

Next step:
  → 07_deployment_artifacts.ipynb
""")
print("=" * 60)